# Useable fields: checking the hypotheses behind each exclusion


The export offers 65 raw fields; 53 survive as useable and 68 columns are
published. Every field kept out along the way was kept out for a stated reason --
"never populated", "always empty", "mostly nonsense", "not usable as stated" --
recorded as a comment in `schema.DROPPED_FIELDS` and `schema.CONSUMED_FIELDS`.
Those comments are hypotheses about the source. They were true when written, and
the source changes weekly.

This notebook checks them. One assert per exclusion, phrased so that it fails
when the reason stops holding -- when a dead field starts carrying data, when a
constant field starts varying. That is the alarm worth having: a field excluded
for a reason that has expired is data we are silently discarding.

Three groups:

1. **Dropped before publishing** -- not in the table at all (`DROPPED_FIELDS`,
   `CONSUMED_FIELDS`).
2. **Published, but carrying almost no information** -- kept in the table because
   they cost nothing and may start varying, but they cannot distinguish one
   programme from another, so nothing downstream should be built on them.
3. **Excluded from a specific reading** -- columns that look like they answer a
   question they do not answer, and rows that hold no data at all.

Bounds are bands, not measured values: an exact-equality notebook would be red
every Monday for no reason. Every check prints what it saw next to what it
required.

In [ ]:
from pathlib import Path

import polars as pl

from fdb_scraper import collect, decode, scrape
from fdb_scraper.config import DROPPED_FIELDS
from fdb_scraper.config import CONSUMED_FIELDS, PIVOTS

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPORT_DIR = ROOT / "data/foerderprogramme_export"  # extracted; omit to download

raw = scrape(export_dir=EXPORT_DIR)          # all 65 raw fields, export names
df = collect(export_dir=EXPORT_DIR)          # the published table
decoded = decode(raw)                        # values decoded, export names kept
N = df.height

print(f"raw       : {raw.height} x {len(raw.columns)}")
print(f"published : {df.height} x {len(df.columns)}")
print(f"\ndropped in process : {sorted(DROPPED_FIELDS)}")
print(f"consumed, not published : {sorted(CONSUMED_FIELDS)}")

In [ ]:
def fill(col: str, frame: pl.DataFrame | None = None) -> int:
    """Rows where the column says something. Empty list == says nothing."""
    frame = df if frame is None else frame
    if isinstance(frame.schema[col], pl.List):
        return frame.select(pl.col(col).list.len().fill_null(0).gt(0).sum()).item()
    return frame.select(pl.col(col).is_not_null().sum()).item()


def distinct(col: str, frame: pl.DataFrame | None = None) -> int:
    """Distinct non-null values; for list columns, distinct elements."""
    frame = df if frame is None else frame
    if isinstance(frame.schema[col], pl.List):
        return frame.select(pl.col(col).explode(empty_as_null=False).drop_nulls().n_unique()).item()
    return frame[col].drop_nulls().n_unique()

## 1. Dropped before publishing

`schema.DROPPED_FIELDS` holds thirteen fields, never parsed at all.
`schema.CONSUMED_FIELDS` holds one more -- `externer_link`, which is parsed
because `add_links` resolves it into `further_links`, then dropped.
Each is checked against `raw`, where they still exist.

### E1 -- seven text fields are never populated

Stated reason: "Null dtype across every programme". The generic CMS template
declares them; the Förderdatenbank editors never fill them.

In [ ]:
NEVER_FILLED = [
    "challenge", "customer_benefit", "proc_quality", "requirements",
    "service_description", "service_fee_descr", "terms_of_payment",
]
for c in NEVER_FILLED:
    print(f"{c:22s} filled {fill(c, raw):5d} / {raw.height}")

# The reason is absolute, so the assert is too: one filled value makes the field
# worth publishing, and the contract check would not catch it (the property exists,
# it is just empty).
assert all(fill(c, raw) == 0 for c in NEVER_FILLED), {
    c: fill(c, raw) for c in NEVER_FILLED
}

**Holds.** All seven empty on all 2500 programmes. Note what this check is for:
`contract.py` verifies the *property* is still declared with the container it
always had, which stays true whether or not anyone types into it. Emptiness is a
value-level fact and only this kind of check sees it.

### E2 -- `foerdertermin` is always an empty link list

Stated reason: "Always an empty link list". A Fördertermin (deadline) classifier
exists in the export's category tree but no programme links one -- worth
distinguishing from E1, since here the column is a list, so "empty" means length
zero rather than null.

In [ ]:
lens = raw.select(pl.col("foerdertermin").list.len().fill_null(0).alias("n"))
print(lens.to_series().value_counts(sort=True))
print(f"total links across all programmes: {lens.to_series().sum()}")

assert lens.to_series().sum() == 0, lens.to_series().sum()

**Holds.** Zero links. Deadlines do reach the table, but as prose in `deadlines`
(the Zusatzinfos sub-section), not as structured dates -- so no column in the
published table answers "which programmes close before X" without parsing German
free text first.

### E3 -- `languages` says "Deutsch" and nothing else

Stated reason: "Single value ('Deutsch') on all but three programmes". A field
with one value cannot separate anything.

In [ ]:
print(raw.select(pl.col("languages").explode(empty_as_null=False)).to_series().value_counts(sort=True))

assert distinct("languages", raw) == 1, distinct("languages", raw)
# The three exceptions are absences, not a second language -- and they are the same
# three stub documents that E12 is about.
assert raw.height - fill("languages", raw) == 3

**Holds.** One value on 2497 programmes, absent on 3 -- and those 3 are the empty
stub documents from E12, not programmes in another language.

### E4 -- `date_of_expiration` is mostly nonsense

Stated reason: "of 638 non-null values, 480 fall outside any plausible range (the
minimum is year 0207)". The most consequential exclusion in the pipeline: an
expiry date is the one thing that would let a consumer tell a live programme from
a dead one, so if the values were usable they would have to be published.

In [ ]:
from datetime import datetime, timezone

exp = raw["date_of_expiration"].drop_nulls()
years = exp.dt.year()
plausible = exp.filter((years >= 2019) & (years <= 2035))
print(f"non-null   : {exp.len()} / {raw.height}")
print(f"min        : {exp.min()}")
print(f"max        : {exp.max()}")
print(f"plausible  : {plausible.len()}  (2019-2035)")
print(f"implausible: {exp.len() - plausible.len()}")
print(raw.select(pl.col("date_of_expiration").dt.year().alias("year")).drop_nulls()
         .to_series().value_counts(sort=True).sort("year").head(12))

# Two independent defects, either one disqualifying: it is absent for three
# quarters of programmes, and where present it is wrong more often than right.
assert exp.len() / raw.height < 0.35, exp.len() / raw.height
assert (exp.len() - plausible.len()) / exp.len() > 0.5
assert exp.min() < datetime(1900, 1, 1, tzinfo=timezone.utc), exp.min()

**Holds.** 638 values of 2500, and 480 of those outside 2019-2035 -- the minimum
is year 0207, so these are not merely stale dates but corrupt ones. Even the 158
plausible values cannot be used: nothing at the row level distinguishes a real
expiry from a mistyped one, so a published column would look authoritative and be
wrong three times in four.

That leaves `status_note` as the export's only lifecycle signal, and E10 shows how
little it carries.

### E5 -- `unternehmensalter` mixes year labels with buckets

Stated reason: "Mix of year labels ('01'..'20') and buckets ('nicht_relevant'),
dominated by 'nicht_relevant'; not usable as stated". Worth checking the shape
closely, because "not usable *as stated*" implies something could be recovered.

In [ ]:
import re

ages = raw.select(
    pl.col("unternehmensalter").list.eval(pl.element().str.extract(r"([^/]+)$")).alias("ua")
)["ua"].to_list()

filled = [a for a in ages if a]
only_nicht_relevant = [a for a in filled if set(a) == {"nicht_relevant"}]
with_years = [a for a in filled if any(re.fullmatch(r"\d{2}", x) for x in a)]


def contiguous(codes: list[str]) -> bool:
    years = sorted(int(x) for x in codes if re.fullmatch(r"\d{2}", x))
    return years == list(range(years[0], years[-1] + 1))


print(f"filled                    : {len(filled)} / {raw.height}")
print(f"only 'nicht_relevant'      : {len(only_nicht_relevant)}")
print(f"carrying year labels       : {len(with_years)}")
print(f"  of those, contiguous runs: {sum(contiguous(a) for a in with_years)}")
print(f"\nexample: {sorted(with_years[0])}")

# Dominated by the placeholder ...
assert len(only_nicht_relevant) / len(filled) > 0.85
# ... so whatever is recoverable applies to a few percent of programmes.
assert len(with_years) / raw.height < 0.06, len(with_years) / raw.height

**Holds.** 1272 programmes fill the field; 1151 of them
say only `nicht_relevant`. Just 100 carry year labels -- and those are contiguous
runs (`01..05` plus `kleiner_als_eins`), i.e. an age *range* expressed as one
category per year.

So the field is not junk, it is a range in disguise: it could be reduced to
min/max company age. Worth writing down, but it stays excluded, because it would
carry an age range for 4% of the table and be silent about the rest -- and a
column that is meaningful on 100 rows out of 2500 is read as meaningful on all of
them.

### E6 -- `comment` is editorial workflow metadata

Stated reason: "Internal editorial ticket ids ('# 861223')". It is non-null on
2495 of 2500 rows, so this is not a sparse field being dropped -- it is a
well-populated field being dropped because of what it contains.

In [ ]:
comment = raw["comment"].drop_nulls()
ids_only = comment.str.contains(r"^[\d\s#/,.-]+$").sum()
ticketish = comment.str.contains(r"(?i)ticket|#").sum()
prose = comment.filter(comment.str.contains(r"[A-Za-zÄÖÜäöüß]{5,}"))

print(f"non-null      : {comment.len()} / {raw.height}")
print(f"distinct      : {comment.n_unique()}")
print(f"ids only      : {ids_only}")
print(f"ticket-shaped : {ticketish}")
print(f"containing prose: {prose.len()}")
print("\nsamples:")
for v in comment.head(4):
    print(" -", v[:120])
print("\nthe prose ones are editorial instructions, not programme description:")
for v in prose.filter(prose.str.len_chars() > 80).head(2):
    print(" -", v[:220])

# Every value carries a number; none is a description of the programme.
assert comment.str.contains(r"\d").all()
# Distinct per row, so it is a per-document workflow note rather than a category.
assert comment.n_unique() == comment.len()

**Holds, and the field is a little more than ids.** Every value contains digits
and every value is unique, so `comment` is per-document bookkeeping: 1594 are pure
ids, 980 name a ticket. Some are German editorial instructions ("Das alte Dokument
sollte bei Publikation des vorliegenden neuen Dokuments depubliziert werden") --
still internal process notes about the *document*, not information about the
programme, so the exclusion stands on the same grounds either way. Publishing it
would only make sense as provenance, and then only if the CC BY-ND terms cover
republishing editor notes -- a licence question, not a data one.

### E7 -- `externer_link` and `path` say it worse than another column does

Two exclusions on different grounds from the rest: both fields hold data, and
another column holds the same thing in a form a consumer can act on.

In [ ]:
print(f"externer_link non-empty : {fill('externer_link', raw)} / {raw.height}")
print(f"further_links non-empty : {fill('further_links')} / {N}")
print("\nraw href :", raw["externer_link"].drop_nulls().filter(
    raw["externer_link"].list.len() > 0)[0][0])
print("published:", df["further_links"].drop_nulls().filter(
    df["further_links"].list.len() > 0)[0][0])
print("\npath     :", raw["path"][0])

# further_links is externer_link resolved: same rows, minus the handful of hrefs
# pointing into the internal training tree, plus a url and title per link.
assert fill("further_links") <= fill("externer_link", raw)
assert fill("externer_link", raw) - fill("further_links") < 50
# path is a temporary extraction directory; url is the stable identifier.
assert str(EXPORT_DIR) in raw["path"][0]

**Holds.** `externer_link` holds `target:/BMWI/...` document references, which are
useless to a consumer; `further_links` is the same list resolved to
`{url, title}`, on 2350 of the same rows. `path` points inside a temporary
extraction directory that ceases to exist after the run.

## 2. Published, but carrying almost no information

These columns are in the table. They were not dropped, because a column costs
close to nothing and a field that is empty today may be filled tomorrow -- but
the hypothesis in each case is that the column cannot distinguish programmes,
either because it is almost always absent or because it is almost always the same
value. Anyone selecting columns for an analysis should know that before spending
time on them.

### E8 -- seven columns are near-empty

Below 3% filled, so any statement made from them describes a few dozen
programmes and says nothing about the other 2400.

Four columns have left this list since it was written. `uf_messen_ausstellungen`
and `uf_aussenwirtschaft` are no longer columns -- `process.collapse_pivots` folds
each pivoted taxonomy into one path column, and neither result is sparse (E11).
`external_id` and `reference_customer` are no longer published at all (E15), and
`functions` is now `application_language` (E15).


In [ ]:
SPARSE = [
    "contact_info_state", "contact_info_mobile", "contact_info_country",
    "contact_info_building", "contact_info_post_box",
    "application_language", "processing_time",
]
for c in SPARSE:
    print(f"{c:30s} {fill(c):5d}  {fill(c) / N:5.1%}")

assert all(fill(c) / N < 0.03 for c in SPARSE), {c: fill(c) for c in SPARSE}
# Nothing outside the list is that sparse, so the list is the whole story.
missed = [c for c in df.columns if fill(c) / N < 0.03 and c not in SPARSE]
assert not missed, missed

# The 21 columns this used to include are two columns now, neither sparse.
for target in PIVOTS:
    print(f"\n{target:30s} {fill(target):5d}  {fill(target) / N:5.1%}"
          f"   (from {len(PIVOTS[target])} source columns)")
    assert fill(target) / N > 0.35, (target, fill(target) / N)


**Holds.** All seven below 3%, and no eighth -- `seo_description` (3.2%) sits just
above the line and is no more useful for it.

`contact_info_state` (1 row) and `contact_info_country` (16) are worth singling
out. The export's `Adresse` has no usable country element, yet contacts in
Brussels and Luxembourg are in there -- so a consumer cannot assume every address
is German, and cannot find the exceptions from these columns either.

`funding_subarea` (91%) and `applicant_sector` (51%) are the counter-examples
that make the threshold worth stating carefully: fill rate measures a column, not
a concept. Pivoted across 19 and 2 source columns the same data looked like
3-18% and 11-48% each, which is why two of those columns used to appear here.


### E9 -- two flags never vary, two categories barely do

`should_not_be_indexed` and `subtype` read like flags that split the table. The
hypothesis is that neither ever takes a second value, so neither splits anything —
which is why neither is published any more, and why this check reads `raw` rather
than `df`. `grw` and `unternehmensgroesse` do vary, but so little that a subset
selected by them is nearly the whole table.


In [ ]:
for c in ("should_not_be_indexed", "subtype"):
    print(c, raw[c].value_counts(sort=True).to_dicts())
for c in ("grw", "unternehmensgroesse"):
    print(c, df.select(pl.col(c).explode(empty_as_null=False)).to_series().value_counts(sort=True).to_dicts())

# should_not_be_indexed is 0-or-absent: the export never sets it. Dropped for that
# reason, but watched here -- a 1 appearing upstream would be the first
# machine-readable "this programme is dead" flag the export has ever offered,
# which given E4 is the one thing we most lack.
assert set(raw["should_not_be_indexed"].drop_nulls().unique()) == {0}
assert raw["subtype"].drop_nulls().n_unique() == 1
# subtype's only usable feature was its null, and that is exactly title's null.
# Joined on url, which both frames carry -- raw is the pre-selection frame.
assert set(df.filter(pl.col("title").is_null())["url"]) == set(
    raw.filter(pl.col("subtype").is_null())["url"]
), "subtype null no longer coincides with title null"


In [ ]:
grw_yes = df.filter(pl.col("grw").list.contains("grw_foerderung")).height
print(f"grw_foerderung: {grw_yes} / {N} ({grw_yes / N:.1%}) -- a boolean, not a category")
assert grw_yes / N < 0.05, grw_yes / N

sizes = df.filter(pl.col("unternehmensgroesse").list.len() > 0).select(
    pl.col("unternehmensgroesse").list.len()
)
print(sizes.to_series().value_counts(sort=True).sort("unternehmensgroesse"))
print(f"filled: {fill('unternehmensgroesse')} / {N}")
print(f"median brackets listed where filled: {sizes.to_series().median()}")
# Where stated at all, most programmes name every bracket, so selecting on company
# size drops almost no rows -- the column looks discriminating and is not.
assert sizes.to_series().median() >= 3

**Holds.** `should_not_be_indexed` is 0 on 2023 rows and absent on 477 -- never 1,
so it is not the retirement flag its name suggests, and the 0-vs-absent split only
records whether the CMS wrote the property. `subtype` is one value on every
non-stub row, and its nulls are exactly the three rows where `title` is null, so
the one predicate it offered duplicates the one E12 already recommends. Both are
dropped; the asserts above read `raw`, so the tripwire survives the drop.

`grw` is 97% "keine_grw_foerderung", a boolean that is false almost always.
`unternehmensgroesse` is filled for 57% and names all four brackets on 959 of
those, so restricting by it drops almost no rows.

`should_not_be_indexed` is the one to watch rather than forget: the day upstream
starts setting it, the export has a retirement flag, which is what E4 shows we
otherwise lack entirely.


## 3. Excluded from a specific reading

Not exclusions of a whole column, but of an interpretation -- two columns that
answer a narrower question than their name suggests, two that are not the keys
they look like, and rows that carry no data at all.

### E10 -- `status_note` is not a lifecycle column

Only 9% of programmes have one, so it cannot describe the table. The stronger
claim is directional: if every filled value states a restriction, then absence
means *unknown*, never *open*, and the 2274 programmes without a note cannot be
counted as active.

In [ ]:
notes = df["status_note"].drop_nulls()
with pl.Config(fmt_str_lengths=100, tbl_rows=-1):
    print(notes.value_counts(sort=True))

# Normalised, because upstream ships akktiv / Antagstellung / merh.
norm = notes.str.to_lowercase()
CAVEAT = r"nicht|ausgelaufen|abgelaufen|beendet|eingestellt|überarbeitung|erstellung|ausschließlich"
print(f"\nfilled: {notes.len()} / {N} ({notes.len() / N:.0%}), distinct: {notes.n_unique()}")
print(f"stating a restriction: {norm.str.contains(CAVEAT).sum()} / {notes.len()}")

assert notes.len() / N < 0.15, notes.len() / N
assert notes.n_unique() < 25, notes.n_unique()
# The load-bearing one: not a single note asserts plain availability.
assert norm.str.contains(CAVEAT).all(), notes.filter(~norm.str.contains(CAVEAT)).to_list()

**Holds.** All 226 filled notes state a restriction -- "Antragstellung nicht mehr
möglich" (167), "derzeit nicht möglich" (44), "ausgelaufen" (5), plus editorial
"in Überarbeitung". Readable per programme; excluded as a lifecycle column, and
excluded as evidence of anything at all for the 2274 programmes without one. The
15 distinct strings include four misspellings of the same sentence, so any
classification must normalise first.

### E11 -- a pivoted taxonomy is not a strict sub-level of what it pivots on

Two taxonomies arrive spread across one column per parent value, and
`process.collapse_pivots` republishes each as one column of `"parent.child"` paths:
`funding_subarea` (19 source columns, pivoted on `funding_area`) and
`applicant_sector` (2, pivoted on `eligible_applicants`).

The obvious reading is a two-level taxonomy where a child implies its parent, so
that grouping the second level up into the first is safe, and the parent need not
be stored. The hypothesis is that neither holds:

1. children occur whose parent the programme does not list, so a roll-up drops
   them silently;
2. a bare child code could not be attributed back anyway -- the child vocabulary
   is shared between parents, and most programmes list several parents.


In [ ]:
from fdb_scraper.config import PIVOT_PARENT_VOCAB, SEPARATOR
from fdb_scraper.schema import pivot_paths

TARGET, PIVOTED_ON = "funding_subarea", "funding_area"

# One row per (parent, child) assignment, so the hierarchy can be checked
# without unpivoting anything.
assign = (
    df.select("id_url", PIVOTED_ON, TARGET)
    .explode(TARGET, empty_as_null=False)
    .drop_nulls(TARGET)
    .with_columns(
        pl.col(TARGET)
        .str.split_exact(SEPARATOR, 1)
        .struct.rename_fields(["parent", "child"])
        .alias("pair")
    )
    .unnest("pair")
    .with_columns(
        orphan=~pl.col(PIVOTED_ON).list.contains(pl.col("parent")).fill_null(False)
    )
)
orphans, total = assign["orphan"].sum(), assign.height
print(f"assignments: {total}, orphaned: {orphans} ({orphans / total:.1%})")
with pl.Config(tbl_rows=-1, fmt_str_lengths=48):
    print(assign.filter("orphan").group_by("parent").len().sort("len", descending=True))

# Small enough that the hierarchy is clearly the intent, large enough that anyone
# grouping child under parent categories loses rows without noticing.
assert 0 < orphans / total < 0.05, orphans / total


In [ ]:
# Claim 2: the parent is not recoverable from the child code alone.
shared = (
    pl.DataFrame({"path": pivot_paths(TARGET)})
    .with_columns(
        pl.col("path")
        .str.split_exact(SEPARATOR, 1)
        .struct.rename_fields(["parent", "child"])
        .alias("pair")
    )
    .unnest("pair")
    .group_by("child")
    .agg(pl.col("parent").n_unique().alias("parents"))
    .filter(pl.col("parents") > 1)
    .sort("parents", descending=True)
)
print(f"{len(pivot_paths(TARGET))} (parent, child) pairs; "
      f"child codes under more than one parent: {shared.height}")
with pl.Config(tbl_rows=-1):
    print(shared)

# What a bare code would cost: values that collapse when the parent is dropped.
bare = assign.select("id_url", "child").unique().height
print(f"\nassignments {assign.height} -> {bare} if the parent is dropped "
      f"({assign.height - bare} lost)")
assert shared.height > 5, shared.height
assert assign.height - bare > 300, assign.height - bare
# And most programmes list several parents, so it could not be guessed either.
multi = df.filter(pl.col(PIVOTED_ON).list.len() > 1).height
print(f"programmes with more than one {PIVOTED_ON}: {multi} / {N} ({multi / N:.0%})")
assert multi / N > 0.5, multi / N

# The same argument, for the other pivot: 15 of the 206 programmes that name a
# sector for both applicant types name *different* sectors -- a doctor opening a
# practice is freie_berufe, an existing medical business is dienstleistungen.
sec = (
    df.select("id_url", "applicant_sector")
    .explode("applicant_sector", empty_as_null=False)
    .drop_nulls("applicant_sector")
    .with_columns(
        pl.col("applicant_sector")
        .str.split_exact(SEPARATOR, 1)
        .struct.rename_fields(["applicant", "sector"])
        .alias("pair")
    )
    .unnest("pair")
)
per_applicant = sec.group_by("id_url").agg(
    pl.col("applicant").n_unique().alias("applicants"),
    pl.col("sector").n_unique().alias("sectors"),
    pl.len().alias("pairs"),
)
differing = per_applicant.filter(
    (pl.col("applicants") > 1) & (pl.col("pairs") > pl.col("sectors"))
).height
both = per_applicant.filter(pl.col("applicants") > 1).height
print(f"\napplicant_sector: {both} programmes name sectors for both applicant "
      f"types, {both - differing} of them the same set")
assert differing > 0, differing


In [ ]:
from fdb_scraper.vocab import CLOSED_VOCABS

parents = set(PIVOTS[TARGET].values())
no_second_level = sorted(set(CLOSED_VOCABS[PIVOT_PARENT_VOCAB[TARGET]]) - parents)
print("first-level categories with no sub-area at all:", no_second_level)
for area in no_second_level:
    k = df.filter(pl.col(PIVOTED_ON).list.contains(area).fill_null(False)).height
    print(f"  {area:24s} {k:5d} programmes")

assert len(no_second_level) == 4, no_second_level
# Three of the four are well populated, so "no second level" is an upstream fact
# rather than an artefact of rarity -- so the second level covers the table unevenly.
sizes = sorted(
    df.filter(pl.col(PIVOTED_ON).list.contains(a).fill_null(False)).height
    for a in no_second_level
)
assert sum(k > 100 for k in sizes) == 3, sizes


**Holds, on both claims.** 132 of 5963 sub-area assignments (2.2%) sit on a
programme that does not list the matching Förderbereich, so rolling the second
level up into the first drops them -- and one programme carries a sub-area with no
`funding_area` at all. The two levels are separate and must be read that way.

The parent is also not recoverable after the fact. 11 sub-area codes occur under
more than one parent -- `beratung_schulung` and `forschung_innovation` under five
each -- and 56% of programmes list several Förderbereiche, so a bare code cannot
be attributed. Dropping the parent would collapse 349 assignments.

`applicant_sector` fails the same way for a different reason: the eight-sector
vocabulary is shared between the two applicant types, and 15 of the 206
programmes that name sectors for both name *different* ones. "Niederlassung von
Ärztinnen und Ärzten" is `freie_berufe` for a founder and `dienstleistungen` for
an existing business; InvestEU lists all eight for founders against one for
companies. So the applicant type has to travel in the value too, even though
`eligible_applicants` also records it.

`corona`, `digitalisierung`, `mobilitaet` and `smart_cities_regionen` have no
second level at all -- the four newest Förderbereiche, never extended to the
sub-taxonomy. Three are substantial (137-212 programmes); `corona` is down to 2,
which is its own signal about how the export ages.


### E12 -- three rows carry no content at all

Nothing in the pipeline drops a programme: a stub document in the export becomes
a row like any other, and 2500 rows is the count everything downstream trusts.
Whether to exclude these three is the consumer's call, so it has to be a
documented, checkable count rather than a surprise.

In [ ]:
CONTENT = [
    "title", "description", "short_description", "legal_basis", "legal_requirements",
    "procedure", "deadlines", "processing_time", "required_documents", "keywords",
]
empty = df.filter(~pl.any_horizontal(pl.col(c).is_not_null() for c in CONTENT))
with pl.Config(fmt_str_lengths=60):
    print(empty.select("programme_slug", "url", "title", "funding_type", "date_of_issue"))

assert empty.height == 3, empty.height
# Empty means empty: no categories either, so no category test excludes them.
assert empty.select(pl.col("funding_area").list.len().fill_null(0).sum()).item() == 0
# A null title is the cheapest test for them, and it catches exactly these rows.
assert df["title"].null_count() == empty.height

**Holds.** Three documents carry a URL and nothing else -- no title, no subtype,
no categories. They are real pages upstream, so this is not a parse failure, and
dropping them in the scraper would hide an upstream problem. Consumers should skip
rows where `title` is null; that one test is exact.

### E13 -- neither `programme_slug` nor `title` is a key

`programme_slug` is the export document's `@name`, which is the final path segment
of a programme's URL, so two programmes in different funding levels can carry the
same one. The hypothesis is that they do -- which rules it out for joining,
deduplicating or counting, and is why the column is not called `id`.


In [ ]:
KEYS = ["programme_slug", "id_url", "id_hash", "url"]
keys = pl.DataFrame({
    "column": KEYS,
    "distinct": [df[c].n_unique() for c in KEYS],
}).with_columns(unique=pl.col("distinct") == N)
print(keys)

dupes = (
    df.group_by("programme_slug").len().filter(pl.col("len") > 1)
    .sort("len", descending=True)
)
print(f"\n{dupes.height} slugs shared by {dupes['len'].sum()} rows")
with pl.Config(fmt_str_lengths=100):
    print(
        df.filter(pl.col("programme_slug") == dupes["programme_slug"][0])
        .select("programme_slug", "url")
    )

assert df["programme_slug"].n_unique() < N, "unique now -- this exclusion has no premise"
# id_url keeps the whole path, so it separates what programme_slug merges.
assert df["id_url"].n_unique() == N
assert df["id_hash"].n_unique() == N and df["url"].n_unique() == N


**Holds.** 22 slugs are shared by 48 rows --
`agrarinvestitionsfoerderungsprogramm` runs in Hessen, Mecklenburg-Vorpommern and
Sachsen-Anhalt as three programmes with three legal bases, and `id` cannot tell
them apart. `id_url` keeps the whole path
(`land-hessen-agrarinvestitionsfoerderungsprogramm`) and is unique across all 2500
rows, as are `id_hash` and `url`. Those three are the keys; `id` is a label.

`title` is the other tempting identifier, because it is the only column a person
can read. Two things have to be true before it could be used to identify a
programme, join two sources on one, or count distinct programmes: titles must be
unique, and where they repeat the repetition must be an artefact of formatting
rather than two genuinely different programmes.

In [ ]:
title = df["title"].drop_nulls()
dup_titles = (
    df.filter(pl.col("title").is_not_null())
    .group_by("title").len().filter(pl.col("len") > 1)
    .sort("len", descending=True)
)
print(f"non-null : {title.len()} / {N}")
print(f"distinct : {title.n_unique()}")
print(f"repeated : {dup_titles.height} titles across {dup_titles['len'].sum()} rows, "
      f"largest group {dup_titles['len'].max()}")
with pl.Config(fmt_str_lengths=80, tbl_rows=8):
    print(dup_titles.head(6))

assert title.n_unique() < title.len(), "titles are unique now"

In [ ]:
# Not a formatting artefact: normalising case and whitespace merges nothing further,
# so the repeated titles are byte-identical strings.
norm = (
    pl.col("title").str.strip_chars().str.to_lowercase().str.replace_all(r"\s+", " ")
)
normalised = df.filter(pl.col("title").is_not_null()).select(norm.alias("t"))["t"]
print(f"distinct titles            : {title.n_unique()}")
print(f"distinct after normalising : {normalised.n_unique()}")
assert normalised.n_unique() == title.n_unique()

# And not duplicate records either: the rows behind a repeated title differ in
# content, so collapsing on title would merge distinct programmes.
same_description = (
    df.filter(pl.col("title").is_in(dup_titles["title"].implode()))
    .group_by("title").agg(pl.col("description").n_unique().alias("n"))
    .filter(pl.col("n") <= 1)
)
print(f"\nrepeated titles whose rows share one description: "
      f"{same_description.height} / {dup_titles.height}")
assert same_description.height / dup_titles.height < 0.2

In [ ]:
# The largest group: one programme per Bundesland, same name, same funding body.
with pl.Config(fmt_str_lengths=45, tbl_rows=20):
    print(
        df.filter(pl.col("title") == dup_titles["title"][0])
        .select("title", "programme_slug", "funding_location", "funding_body")
    )

# Adding the region does not rescue it -- title + funding_location still collides.
located = df.filter(pl.col("title").is_not_null()).with_columns(
    pl.col("funding_location").list.sort().list.join("|").alias("loc")
)
pairs = located.select("title", "loc")
print(f"\ndistinct (title, funding_location): {pairs.n_unique()} of {located.height}")
with pl.Config(fmt_str_lengths=55):
    print(
        located.join(
            pairs.group_by("title", "loc").len().filter(pl.col("len") > 1).drop("len"),
            on=["title", "loc"],
        ).select("title", "loc", "programme_slug").sort("title")
    )

assert pairs.n_unique() < located.height

**Holds, and the failure is not the kind you can normalise away.** 2497 titles,
2448 distinct: 28 titles are shared by 77 rows. Normalising case and whitespace
merges nothing, so these are byte-identical strings, and only 2 of the 28 groups
have rows that even share a description -- the other 26 are distinct programmes
that happen to be named the same.

The largest group is "Förderung im Rahmen des Denkmalschutz-Sonderprogramms": 16
rows, one per Bundesland, all with `funding_body = bund`, distinguished only by
`funding_location` and by the `-BW` / `-BY` / `-BE` suffix in `id`. That pattern
suggests title plus region might work as a composite key, and it does not: 3 pairs
still collide, including two Hamburg programmes both called "Förderung für den
Neubau von Wohnungen für Studierende und Auszubildende".

So the name is a display value. Anything that identifies, joins or counts
programmes has to use `id_url`, `id_hash` or `url`, and a count of distinct titles
understates the table by 49.

### E14 -- `applicant_sector` is conditional on the applicant type

`applicant_sector` is filled on 51% of programmes, and its two halves are very
unevenly used: 4738 pairs name a sector for `unternehmen` against 1352 for
`existenzgruenderin`. Read as a fill rate that looks like patchy data.

It is not. The claim is directional, and it is the one that matters for reading
an absence: if a sector nearly always sits on a programme that lists that
applicant type as eligible, then a missing `existenzgruenderin.*` means "not a
founder programme", not "founder programme of unknown sector".


In [ ]:
pairs = (
    df.select("id_url", "eligible_applicants", "applicant_sector")
    .explode("applicant_sector", empty_as_null=False)
    .drop_nulls("applicant_sector")
    .with_columns(
        pl.col("applicant_sector")
        .str.split_exact(SEPARATOR, 1)
        .struct.rename_fields(["applicant", "sector"])
        .alias("p")
    )
    .unnest("p")
    .with_columns(
        targeted=pl.col("eligible_applicants")
        .list.contains(pl.col("applicant"))
        .fill_null(False)
    )
)
with pl.Config(tbl_rows=-1):
    print(
        pairs.group_by("applicant").agg(
            pl.len().alias("pairs"),
            pl.col("targeted").sum().alias("applicant_is_eligible"),
            pl.col("sector").n_unique().alias("sectors"),
        ).sort("pairs", descending=True)
    )
print(f"\nfilled: {fill('applicant_sector')} / {N} "
      f"({fill('applicant_sector') / N:.1%})")

# The load-bearing one: a sector almost never appears for an applicant type the
# programme does not list, so an absent half is "does not apply", not "unknown".
share = pairs["targeted"].mean()
print(f"pairs whose applicant type is also in eligible_applicants: {share:.1%}")
assert share > 0.95, share
# Both halves use the whole sector vocabulary, so neither is a token entry.
assert pairs.group_by("applicant").agg(pl.col("sector").n_unique())["sector"].min() == 8


**Holds.** 5988 of 6090 pairs (98.3%) name an applicant type the programme also
lists as eligible, and both halves use all eight sectors, so neither is a token
entry. The uneven split is the applicant types being unevenly common -- 1504
programmes admit `unternehmen` against 286 admitting `existenzgruenderin` -- not
the field being neglected.

Two caveats an analysis has to carry. The relationship is not an implication in
either direction: 102 pairs name a sector for an applicant type that is not in
`eligible_applicants`, and plenty of programmes admit an applicant type without
naming any sector -- so `applicant_sector` cannot be used to *find* programmes for
a given applicant, and `eligible_applicants` remains the column for that.

And the applicant type is not decoration on the sector. E11 has the detail: 15 of
the 206 programmes that name sectors for both types name different ones, so the
two halves cannot be pooled into a plain sector list.


## 4. CMS names that describe the wrong thing

The XML property names come from a generic content-management template, and the
"Field names" table in the README already records nine that mean the opposite of
what they say. Four more were only caught by reading the values: two whose content
belongs under a different name, and two that turn out to say nothing the table
does not say better.

Grouped here rather than renumbered into section 1 because they share one cause,
and because the check is the same in each case -- read the values, not the name.

### E15 -- two fields hold something other than their name

`gsb:functions` and `gsb:remark` are both populated, both plausible-sounding, and
neither holds what it says. The hypothesis is that their content is identifiable
as something else entirely -- an application language and a search-engine
description -- which is why they are published as `application_language` and
`seo_description`.

The first matters more than its 1% fill suggests: `languages` was dropped in E3
for saying only "Deutsch", and this is where the exceptions to that ended up.

In [ ]:
lang = df["application_language"].drop_nulls()
seo = df["seo_description"].drop_nulls()
with pl.Config(fmt_str_lengths=95, tbl_rows=-1):
    print(lang.value_counts(sort=True))
print()
for v in seo.head(3):
    print(f"  {v[:95]!r}")

# application_language: every value names a language or talks about one.
LANG = r"(?i)deutsch|englisch|sprache|german|english"
print(f"\napplication_language: {lang.len()} values, {lang.n_unique()} distinct")
assert lang.str.contains(LANG).all(), lang.filter(~lang.str.contains(LANG)).to_list()
# And it says more than the dropped languages classifier could: some programmes
# accept English sketches, which "Deutsch" alone cannot express.
assert lang.str.contains(r"(?i)englisch|english").any()

# seo_description: search-engine copy, so second person and a call to action.
CTA = r"(?i)beantragen|erhalten sie|sichern sie|infos|informieren"
print(f"seo_description     : {seo.len()} values, {seo.n_unique()} distinct")
assert seo.str.contains(CTA).mean() > 0.8, seo.str.contains(CTA).mean()
# Not an editorial note: it never states a caveat the way status_note does.
assert not seo.str.contains(r"(?i)nicht mehr möglich|ausgelaufen").any()


**Holds.** `application_language` has 11 distinct values over 25 programmes and
every one of them names a language or discusses one -- "Deutsch", "Englisch", "Die
Antragstellung ist ausschließlich in deutscher Sprache möglich", "Die
Antragssprache für Skizzen ist in der Regel Englisch". Some do mention English, so
this field carries information the `languages` classifier provably could not: E3
found that classifier says "Deutsch" and nothing else, and dropped it on that
basis. The exceptions were in `gsb:functions` all along.

`seo_description` is search-result copy: second person, imperative, 81 values of
which the vast majority open with "Beantragen Sie" or promise "Infos zu". Not one
states a restriction, which is what separates it from `status_note` (E10) --
`status_note` never asserts availability and this never asserts anything else.
Published under a name that says what it is, so nobody reads it as an editorial
remark about the programme.

### E16 -- two fields say nothing the table does not say better

`gsb:externalID` and `gsb:referenceCustomer` are both filled on about 1% of
programmes. The hypothesis is that neither adds anything: one is unresolvable, the
other duplicates a column that covers 98% of the table instead of 1%.

In [ ]:
ext = raw["external_id"].drop_nulls()
print(f"external_id: {ext.len()} values, {ext.n_unique()} distinct")
print("  ", ext.head(4).to_list())
# Opaque: no prefix, no scheme, no upstream documentation, nothing to join
# against. An identifier nobody can resolve is not an identifier -- and one row
# packs two of them into a comma-separated string, so it is not even one id per
# programme.
assert ext.str.contains(r"^\d{14}(, \d{14})*$").all(), ext.to_list()
multi = ext.str.contains(",").sum()
print(f"  rows carrying more than one id: {multi}")
assert multi == 1, multi
assert ext.len() / N < 0.01, ext.len() / N

# reference_customer names the funding ministry -- which foerderorganisation
# already carries as a code, on every row that has one.
ref = decoded.filter(pl.col("reference_customer").is_not_null())
print(f"\nreference_customer: {ref.height} rows")
with pl.Config(fmt_str_lengths=52, tbl_width_chars=150):
    print(ref.select("reference_customer", "foerderorganisation").head(4))

# The load-bearing one: it is almost always a label for a body the table already
# names, rather than a fact of its own.
also = ref.filter(pl.col("foerderorganisation").list.len() > 0).height
print(f"of those, foerderorganisation also set: {also} / {ref.height}")
with pl.Config(fmt_str_lengths=52, tbl_width_chars=150):
    print(
        ref.filter(pl.col("foerderorganisation").list.len() == 0)
        .select("reference_customer", "kontakt")
    )
# Two exceptions, and only one of them names a body nothing else does: the other
# repeats its own kontakt. So the column is recoverable on 24 of 25 rows.
assert also >= ref.height - 2, (also, ref.height)
# And foerderorganisation covers vastly more of the table.
org = decoded.filter(pl.col("foerderorganisation").list.len() > 0).height
print(f"foerderorganisation filled: {org} / {N} ({org / N:.0%})")
assert org > 20 * ref.height, (org, ref.height)


**Holds, with one row of loss to record.** `external_id` is ten values of 14-digit
numbers -- `99102158080000`, `99400099000000` -- identical in shape, with no
prefix, no scheme and nothing upstream that says which registry they belong to.
One row carries two of them in a comma-separated string
(`99400439017000, 99400439000000`), so the column is not even one identifier per
programme. Unresolvable and unparsed, so dropped rather than published as
something a consumer might try to join on.

`reference_customer` names the funding ministry as a display string. 23 of its 25
rows also carry the same body in `foerderorganisation` as a code:
"Bundesministerium für Wirtschaft und Klimaschutz (BMWK)" against
`bmwe-bundesministerium_wirtschaft_energie`. Of the two exceptions, "Thüringer
Aufbaubank (TAB)" repeats its own `kontakt` -- so 24 of 25 rows are recoverable
without it.

The twenty-fifth is a real loss, and small enough to state exactly: "Förderung der
Teilhabe und Unterstützung älterer Menschen" names the Sächsisches
Staatsministerium für Soziales und Verbraucherschutz here, while its `kontakt`
points at the Sächsische Aufbaubank -- the ministry behind the programme against
the bank administering it. Dropping the column loses that distinction on that one
programme, against carrying a column filled on 1% of the table where
`foerderorganisation` covers 98%. The labels belong in `vocab.py` alongside every
other code's label.

## Summary

| # | Excluded | Reason still holds? |
| --- | --- | --- |
| E1 | 7 text fields | yes -- 0 filled of 2500 |
| E2 | `foerdertermin` | yes -- 0 links |
| E3 | `languages` | yes -- 1 distinct value, and the exceptions live in `application_language` (E15) |
| E4 | `date_of_expiration` | yes -- 638 filled, 480 implausible, min year 0207 |
| E5 | `unternehmensalter` | yes -- 1151 of 1272 say only `nicht_relevant`; the 100 year-runs are a recoverable age range for 4% of the table |
| E6 | `comment` | yes -- ids, tickets and editorial instructions, all about the document |
| E7 | `externer_link`, `path` | yes -- `further_links` and `url` say the same thing usefully |
| E8 | 7 near-empty columns | yes -- all below 3% filled, no eighth |
| E9 | `should_not_be_indexed`, `subtype`, `grw`, `unternehmensgroesse` | yes -- 1 value, 1 value, 97%-one-value, all-brackets-listed. The first two are now dropped |
| E10 | `status_note` as lifecycle | yes -- 226 filled, every one a restriction |
| E11 | a pivoted taxonomy as a strict sub-level | yes -- 132 orphans of 5963, 4 Förderbereiche with no second level, 11 child codes under several parents |
| E12 | 3 stub rows | yes -- exactly 3, all with null `title` |
| E13 | `programme_slug` and `title` as keys | yes -- 22 slugs shared by 48 rows; 28 titles shared by 77 rows, 26 of them distinct programmes |
| E14 | `applicant_sector` as patchy data | yes -- 98.3% of pairs name an applicant type the programme lists as eligible |
| E15 | `functions`, `remark` as their names | yes -- an application language and search-engine copy |
| E16 | `external_id`, `reference_customer` | yes -- unresolvable ids; a ministry label recoverable on 24 of 25 rows |

Two exclusions worth revisiting if the export changes, and now guarded by an
assert that will say so:

* `should_not_be_indexed` (E9) -- the moment a 1 appears, the export has a
  retirement flag, which is what E4 shows we otherwise lack entirely.
* `unternehmensalter` (E5) -- recoverable as a min/max age range; blocked by
  coverage, not by shape.

And one thing the table cannot do at all:

* No structured deadline or expiry column survives anywhere (E2, E4), so whether a
  programme is still open is not answerable from this table. Any answer is an
  inference from `status_note`, and E10 shows that inference is unsound for 91% of
  programmes -- worth stating in the README next to the field list, since it is the
  first thing a consumer will try to compute.
